## **Road Pulse:** Feature Engineering

### Library Imports

In [1]:
import numpy as np
import pandas as pd

### **Loading data**

In [2]:
def load_data():
    loaded_data = []

    for i in range(7):
        data = pd.read_excel(f'../data/raw/nez-opendata-202{i}.xlsx')
        data.columns = ['accident_id', 'department', 'municipality', 'date_time', 'longitude', 'latitude', 'accident_type', 'involved_vehicles_num', 'description']

        data['date_time'] = pd.to_datetime(
            data['date_time'],
            format="%d.%m.%Y,%H:%M", 
            errors='coerce'
        )

        loaded_data.append(data)

    return pd.concat(loaded_data)

In [3]:
data = load_data()

In [ ]:
data.to_excel('../data/raw/nez-opendata-all.xlsx', index=False)

In [4]:
data.head()

,accident_id,department,municipality,date_time,longitude,latitude,accident_type,involved_vehicles_num,description
0,1278576,BEOGRAD,BARAJEVO,2020-01-07 10:10:00,20.301589,44.568563,Sa mat.stetom,SN SA JEDNIM VOZILOM,Nezgoda sa jednim vozilom – silazak sa kolovoz...
1,1278948,BEOGRAD,BARAJEVO,2020-01-09 12:50:00,20.413280,44.579780,Sa mat.stetom,SN SA NAJMANjE DVA VOZILA – BEZ SKRETANjA,Najmanje dva vozila koja se kreću u istom smer...
2,1278074,BEOGRAD,BARAJEVO,2020-01-10 10:05:00,20.312560,44.575470,Sa mat.stetom,SN SA NAJMANjE DVA VOZILA – SKRETANjE ILI PREL...,Najmanje dva vozila koja se kreću istim putem ...
3,1278582,BEOGRAD,BARAJEVO,2020-01-10 15:15:00,20.412190,44.637770,Sa mat.stetom,SN SA NAJMANjE DVA VOZILA – SKRETANjE ILI PREL...,Najmanje dva vozila koja se kreću različitim p...
4,1278638,BEOGRAD,BARAJEVO,2020-01-16 16:45:00,20.381686,44.633030,Sa povredjenim,SN SA NAJMANjE DVA VOZILA – BEZ SKRETANjA,Najmanje dva vozila koja se kreću u istom smer...


### **Feature Engineering**

#### **Feature:** `date_time`

New features:
- `year`
- `month`
- `day_of_week`
- `hour`
- `day_type` (Weekday or Weekend)
- `is_rush`
- `is_night`

In [5]:
data['year'] = data['date_time'].dt.year
data['month'] = data['date_time'].dt.month
data['day_of_week'] = data['date_time'].dt.day_of_week
data['hour'] = data['date_time'].dt.hour

data = data.drop(columns='date_time')

In [6]:
data['day_type'] = np.where((data['day_of_week'] == 5) | (data['day_of_week'] == 6), "Weekend", "Weekday")

In [7]:
data['is_rush'] = data['hour'].isin([7, 8, 16, 17, 18])

In [8]:
data['is_night'] = data['hour'].between(22, 23) | data['hour'].between(0, 6)

In [9]:
data.head(3)

,accident_id,department,municipality,longitude,latitude,accident_type,involved_vehicles_num,description,year,month,day_of_week,hour,day_type,is_rush,is_night
0,1278576,BEOGRAD,BARAJEVO,20.301589,44.568563,Sa mat.stetom,SN SA JEDNIM VOZILOM,Nezgoda sa jednim vozilom – silazak sa kolovoz...,2020,1,1,10,Weekday,False,False
1,1278948,BEOGRAD,BARAJEVO,20.413280,44.579780,Sa mat.stetom,SN SA NAJMANjE DVA VOZILA – BEZ SKRETANjA,Najmanje dva vozila koja se kreću u istom smer...,2020,1,3,12,Weekday,False,False
2,1278074,BEOGRAD,BARAJEVO,20.312560,44.575470,Sa mat.stetom,SN SA NAJMANjE DVA VOZILA – SKRETANjE ILI PREL...,Najmanje dva vozila koja se kreću istim putem ...,2020,1,4,10,Weekday,False,False


#### **Feature:** `accident_type`

In [10]:
mapping = {
    'Sa mat.stetom': 'material',
    'Sa povredjenim': 'injured',
    'Sa poginulim': 'dead',
}

In [11]:
data['accident_type'] = data['accident_type'].map(mapping)

In [12]:
data.groupby('accident_type').size()

accident_type
dead          2979
injured      79828
material    122545
dtype: int64

In [13]:
data['accident_type'] = np.where((data['accident_type'] == 'material') |( data['accident_type'] == 'injured'), 0, 1)

In [14]:
data.groupby('accident_type').size()

accident_type
0    202373
1      2979
dtype: int64

In [15]:
data.head(3)

,accident_id,department,municipality,longitude,latitude,accident_type,involved_vehicles_num,description,year,month,day_of_week,hour,day_type,is_rush,is_night
0,1278576,BEOGRAD,BARAJEVO,20.301589,44.568563,0,SN SA JEDNIM VOZILOM,Nezgoda sa jednim vozilom – silazak sa kolovoz...,2020,1,1,10,Weekday,False,False
1,1278948,BEOGRAD,BARAJEVO,20.413280,44.579780,0,SN SA NAJMANjE DVA VOZILA – BEZ SKRETANjA,Najmanje dva vozila koja se kreću u istom smer...,2020,1,3,12,Weekday,False,False
2,1278074,BEOGRAD,BARAJEVO,20.312560,44.575470,0,SN SA NAJMANjE DVA VOZILA – SKRETANjE ILI PREL...,Najmanje dva vozila koja se kreću istim putem ...,2020,1,4,10,Weekday,False,False


#### **Feature:** `involved_vehicle_number`

In [16]:
data['involved_vehicles_num'].unique()

<StringArray>
[                              'SN SA JEDNIM VOZILOM',
          'SN SA NAJMANjE DVA VOZILA – BEZ SKRETANjA',
 'SN SA NAJMANjE DVA VOZILA – SKRETANjE ILI PRELAZAK',
                          'SN SA PARKIRANIM VOZILIMA',
                                     'SN SA PEŠACIMA']
Length: 5, dtype: str

In [ ]:
def preprocess_involved_vehicles_num(df, column='involved_vehicles_num'):

    mapping = {
        r'SN SA JEDNIM VOZILOM': 'single_vehicle',
        r'SN SA NAJMANjE DVA VOZILA – BEZ SKRETANjA': 'two_vehicles_no_turn',
        r'SN SA NAJMANjE DVA VOZILA – SKRETANjE ILI PRELAZAK': 'two_vehicles_turn_or_cross',
        r'SN SA PARKIRANIM VOZILIMA': 'parked_vehicles',
        r'SN SA PEŠACIMA': 'pedestrians'
    }

    df[column] = df[column].astype(str).str.strip()

    df[column] = df[column].replace(mapping, regex=True)

    return df    

Binary Flag Decomposition:

In [18]:
def extract_involved_vehicles_column(df, column='involved_vehicles_num'):
    columns = pd.DataFrame(index=df.index)

    values = df[column].fillna('').str.lower()

    columns['single_vehicle'] = values.str.contains('jednim vozilom').astype(int)

    columns['multiple_vehicles'] = values.str.contains('najmanje dva vozila').astype(int)
    
    columns['parked_vehicles'] = values.str.contains('parkiranim vozilima').astype(int)

    columns['pedestrian'] = values.str.contains('pešacima').astype(int)

    columns['turning_crossing'] = values.str.contains('spretanje|prelazak').astype(int)

    columns['no_turning'] = values.str.contains('bez skretanja').astype(int)

    return columns

> Instead of 5 categories we create multiple binary categories where each represents one characteristic of the accident.

One-Hot Encoding:

In [19]:
data = preprocess_involved_vehicles_num(df=data)

In [ ]:
dummies = pd.get_dummies(data['involved_vehicles_num'], prefix='acc', dtype=int)
data = pd.concat([data, dummies], axis=1)

In [21]:
data.drop(columns='involved_vehicles_num', inplace=True)
data.head(3)

,accident_id,department,municipality,longitude,latitude,accident_type,description,year,month,day_of_week,hour,day_type,is_rush,is_night,acc__parked_vehicles,acc__pedestrians,acc__single_vehicle,acc__two_vehicles_no_turn,acc__two_vehicles_turn_or_cross
0,1278576,BEOGRAD,BARAJEVO,20.301589,44.568563,0,Nezgoda sa jednim vozilom – silazak sa kolovoz...,2020,1,1,10,Weekday,False,False,0,0,1,0,0
1,1278948,BEOGRAD,BARAJEVO,20.413280,44.579780,0,Najmanje dva vozila koja se kreću u istom smer...,2020,1,3,12,Weekday,False,False,0,0,0,1,0
2,1278074,BEOGRAD,BARAJEVO,20.312560,44.575470,0,Najmanje dva vozila koja se kreću istim putem ...,2020,1,4,10,Weekday,False,False,0,0,0,0,1


#### **Feature:** `description`

In [25]:
data['description'].unique()

<StringArray>
[                                                                                  'Nezgoda sa jednim vozilom – silazak sa kolovoza u krivini',
                                                                  'Najmanje dva vozila koja se kreću u istom smeru – uključivanje u saobraćaj',
                                 'Najmanje dva vozila koja se kreću istim putem u suprotnim smerovima uz skretanje ulevo ispred drugog vozila',
                                  'Najmanje dva vozila koja se kreću različitim putevima uz skretanje udesno ispred vozila koje nailazi sleva',
                                                                                'Najmanje dva vozila koja se kreću u istom smeru – sustizanje',
                                                                            'Nezgoda sa jednim vozilom – silazak udesno sa kolovoza na pravcu',
                                                  'Ostale nezgode sa najmanje dva vozila koja se kreću istim putem u istom

In [31]:
def extract_description(df, column='description'):

    columns = pd.DataFrame(index=df.index)

    values = df[column].fillna('').str.lower()
    
    return values

In [32]:
columns = extract_description(df=data)
data = pd.concat([data, columns], axis=1)

In [21]:
print(f'Number of Features: {len(data.columns)}')
print('Features:')
print(*data.columns, sep='\n')

Number of Features: 19
Features:
accident_id
department
municipality
longitude
latitude
accident_type
description
year
month
day_of_week
hour
day_type
is_rush
is_night
acc_type_SN SA JEDNIM VOZILOM
acc_type_SN SA NAJMANjE DVA VOZILA – BEZ SKRETANjA
acc_type_SN SA NAJMANjE DVA VOZILA – SKRETANjE ILI PRELAZAK
acc_type_SN SA PARKIRANIM VOZILIMA
acc_type_SN SA PEŠACIMA


In [ ]:
data.drop(columns=['accident_id'])

### **Saving preprocessed data**

In [ ]:
data.to_excel('../data/processed/accident_processed_srb.xlsx', index=False)